# QuantFinance-EDU: Full System Demo

This notebook demonstrates all seven modules on a user-provided set of tickers.

**You can change the tickers and dates below.**


In [ ]:
# =============================================================
# CONFIGURATION — CHANGE THESE
# =============================================================
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'XOM']  # Your stocks
START_DATE = '2018-01-01'
END_DATE = None  # None = today
RUN_BACKTEST = True
RUN_SENSITIVITY = False  # Set True for full sensitivity analysis (slower)
# =============================================================

import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

print('Libraries loaded successfully.')
print(f'\nAnalysing: {TICKERS}')
print(f'Period: {START_DATE} → {END_DATE or "today"}')

## Step 1: Data Loading & Validation

Before any analysis, we validate the data quality.
Any problems found here must be resolved before proceeding.


In [ ]:
from src.data.config_loader import get_config
from src.data.fetcher import DataFetcher
from src.data.validator import DataValidator

config = get_config()
fetcher = DataFetcher(config)
validator = DataValidator(config)

# Fetch price data
print('Fetching price data...')
price_data = fetcher.fetch_price_data(TICKERS, start=START_DATE, end=END_DATE)
print(f'  Downloaded data for: {list(price_data.keys())}')
for ticker, df in price_data.items():
    print(f'  {ticker}: {len(df)} days ({df.index[0].date()} → {df.index[-1].date()})')

# Validate
print('\nRunning data quality checks...')
quality_report = validator.generate_quality_report(price_data, {})
validator.print_quality_report(quality_report)

if not quality_report.passed:
    print('\n⚠ Data quality issues found. Review errors above before proceeding.')

## Step 2: Return Calculations and Method Comparison

**Decision 001**: We use log returns for time-series analysis.
This cell shows the mathematical comparison between log and arithmetic returns.

**Mathematical formula:**
```
Log return:      r_t = ln(P_t / P_{t-1})
Arithmetic:      r_t = (P_t - P_{t-1}) / P_{t-1}
```


In [ ]:
from src.preprocessing.returns import (
    log_returns, arithmetic_returns, compare_return_methods, annualised_return
)

ticker = TICKERS[0]
prices = price_data[ticker]['Adj Close']

comparison = compare_return_methods(prices)
print(f'Return Method Comparison for {ticker}:')
print(comparison.to_string())

# Show the time-additivity property of log returns
log_r = log_returns(prices)
monthly_log = log_r.resample('ME').sum()  # Sum of daily log returns = monthly log return
direct_monthly = np.log(prices.resample('ME').last() / prices.resample('ME').first())

corr = monthly_log.corr(direct_monthly)
print(f'\nTime-additivity check for {ticker}:')
print(f'  Sum of daily log returns vs direct monthly log return: correlation = {corr:.6f}')
print(f'  (Should be exactly 1.0 if time-additive)')

In [ ]:
from src.visualisation.plots import Plots

plotter = Plots()

# Plot return distribution for first ticker
log_r = log_returns(price_data[TICKERS[0]]['Adj Close'])
plotter.plot_return_distribution(log_r, TICKERS[0])
plt.show()

## Step 3: Module 1 — Business Quality

**Mathematical framework:**
- **Piotroski F-Score**: F = Σᵢ fᵢ where fᵢ ∈ {0,1}, i=1..9
- **DuPont ROE**: ROE = Net Margin × Asset Turnover × Equity Multiplier  
- **Altman Z**: Z = 1.2·X₁ + 1.4·X₂ + 3.3·X₃ + 0.6·X₄ + 1.0·X₅


In [ ]:
from src.modules.m1_fundamentals import FundamentalsModule

m1 = FundamentalsModule(config)

print('Fetching fundamental data...')
fundamental_data = fetcher.fetch_fundamental_data(TICKERS)

m1_results = {}
for ticker in TICKERS:
    if ticker in fundamental_data:
        result = m1.run(
            ticker=ticker,
            price_data=price_data[ticker],
            fundamental_data=fundamental_data[ticker]
        )
        m1_results[ticker] = result
        print(f'\n{ticker}: Score = {result.normalised_score:.1f}/100')
        print(m1.explain(result))

## Step 4: Module 3 — Time-Series Characteristics

**Mathematical framework:**
- **Momentum**: MOM = r(t-252, t-21) — 12-month return, skip 1 month
- **Hurst Exponent**: H ∈ (0,1): H<0.5 mean-reverting, H=0.5 random walk, H>0.5 trending
- **Autocorrelation**: ρ_k = Cov(r_t, r_{t-k}) / Var(r_t)


In [ ]:
from src.modules.m3_timeseries import TimeSeriesModule

m3 = TimeSeriesModule(config)

m3_results = {}
for ticker in TICKERS:
    result = m3.run(ticker=ticker, price_data=price_data[ticker])
    m3_results[ticker] = result
    print(f'{ticker}: M3 Score = {result.normalised_score:.1f}/100')
    
    h = result.components.get('hurst', {}).get('h_exponent')
    mom = result.components.get('momentum', {}).get(252, {}).get('return')
    if h:
        interp = 'trending' if h > 0.55 else ('mean-reverting' if h < 0.45 else 'random walk')
        print(f'  Hurst = {h:.3f} → {interp}')
    if mom is not None:
        print(f'  12-month momentum = {mom:.1%}')

In [ ]:
# Rolling volatility chart
log_r = log_returns(price_data[TICKERS[0]]['Adj Close'])
plotter.plot_rolling_volatility(log_r, TICKERS[0])
plt.show()

## Step 5: Module 4 — Factor Model

**Mathematical framework (Carhart 4-Factor):**
```
R_i - R_f = α + β₁·(R_m - R_f) + β₂·SMB + β₃·HML + β₄·WML + ε
```

Estimated via OLS: β = (XᵀX)⁻¹ Xᵀy


In [ ]:
from src.modules.m4_factors import FactorModel

print('Downloading Fama-French factor data...')
ff_data = fetcher.fetch_ff_factors(factor_set='4')  # 3 + momentum
print(f'  FF data downloaded: {len(ff_data)} monthly observations')

m4 = FactorModel(config)
m4_results = {}

for ticker in TICKERS:
    rf_data = ff_data[['RF']] if 'RF' in ff_data.columns else None
    result = m4.run(
        ticker=ticker,
        price_data=price_data[ticker],
        ff_factors=ff_data,
        risk_free_rate=rf_data
    )
    m4_results[ticker] = result
    betas = result.components.get('betas', {})
    alpha = result.components.get('alpha', 'N/A')
    r2 = result.components.get('r_squared', 'N/A')
    print(f'\n{ticker}:')
    print(f'  Alpha (monthly) = {alpha:.4f}' if isinstance(alpha, float) else f'  Alpha = {alpha}')
    print(f'  R² = {r2:.3f}' if isinstance(r2, float) else f'  R² = {r2}')
    for factor, beta in betas.items():
        print(f'  β_{factor} = {beta:.3f}')

## Step 6: Module 5 — Risk

**Different dimensions of risk:**

| Measure | What it captures | Formula |
|---------|-----------------|--------|
| Volatility | Average magnitude of fluctuations | σ = std(r) × √252 |
| Downside deviation | Volatility of losses only | σ_d = std(min(r,0)) × √252 |
| Max Drawdown | Worst peak-to-trough decline | MDD = min_t [(V_t - peak_t) / peak_t] |
| VaR 95% | 1-in-20 worst day | -Q(r, 5%) |
| CVaR 95% | Average loss beyond VaR | -E[r \| r < VaR] |


In [ ]:
from src.modules.m5_risk import RiskModule

benchmark_data = fetcher.fetch_benchmark(start=START_DATE, end=END_DATE)
m5 = RiskModule(config)

m5_results = {}
print(f'{"Ticker":8} {"Vol":8} {"MDD":8} {"VaR95":8} {"CVaR95":8} {"Beta":8} {"Sharpe":8}')
print('-' * 60)

for ticker in TICKERS:
    result = m5.run(
        ticker=ticker,
        price_data=price_data[ticker],
        benchmark_data=benchmark_data
    )
    m5_results[ticker] = result
    c = result.components
    vol = c.get('volatility', {}).get('annual_volatility', float('nan'))
    mdd = c.get('max_drawdown', {}).get('max_drawdown', float('nan'))
    var95 = c.get('var', {}).get(0.95, {}).get('var', float('nan'))
    cvar95 = c.get('var', {}).get(0.95, {}).get('cvar', float('nan'))
    beta = c.get('beta', {}).get('beta', float('nan'))
    sharpe = c.get('sharpe', {}).get('sharpe_ratio', float('nan'))
    print(f'{ticker:8} {vol:8.1%} {mdd:8.1%} {var95:8.2%} {cvar95:8.2%} {beta:8.2f} {sharpe:8.2f}')

In [ ]:
# Drawdown chart
plotter.plot_drawdown(price_data[TICKERS[0]]['Adj Close'], TICKERS[0])
plt.show()

## Step 7: Module 7 — Portfolio Optimisation

**Mathematical framework:**

Portfolio expected return: **E(R_p) = wᵀμ**

Portfolio variance: **σ_p² = wᵀΣw**

Sharpe maximisation: **max_w (wᵀμ - r_f) / √(wᵀΣw)**

Subject to: Σwᵢ = 1, wᵢ ≥ 0, wᵢ ≤ 0.40


In [ ]:
from src.modules.m7_optimisation import PortfolioOptimiser

m7 = PortfolioOptimiser(config)

# Compute returns matrix
returns_df = pd.DataFrame({
    ticker: log_returns(price_data[ticker]['Adj Close'])
    for ticker in TICKERS if ticker in price_data
}).dropna()

# Estimate expected returns and covariance
mu = m7.estimate_expected_returns(price_data, method='historical')
sigma = m7.estimate_covariance(returns_df, method='ledoit_wolf')

print('Estimated Annual Expected Returns:')
print(mu.apply(lambda x: f'{x:.2%}'))

print('\nCovariance Matrix (annual):')
print((sigma * 252).round(4))

In [ ]:
# Optimise portfolio
opt_result = m7.optimise_portfolio(mu, sigma, objective='max_sharpe')

print('\nOptimal Portfolio (Max Sharpe):')
print(f'{"Ticker":10} {"Weight":>10}')
print('-' * 22)
for ticker, w in opt_result['weights'].items():
    print(f'{ticker:10} {w:>10.1%}')
print('-' * 22)
print(f'Expected Return: {opt_result["expected_return"]:.2%}')
print(f'Expected Volatility: {opt_result["expected_volatility"]:.2%}')
print(f'Sharpe Ratio: {opt_result["sharpe_ratio"]:.3f}')

# Compare strategies
rf_data = fetcher.fetch_ff_factors('3')
rf = float(rf_data['RF'].mean() / 100) * 252 if rf_data is not None and 'RF' in rf_data else 0.04

strategy_comparison = m7.compare_strategies(returns_df, mu, sigma, risk_free_rate=rf)
print('\nStrategy Comparison:')
print(strategy_comparison.to_string())

In [ ]:
# Efficient frontier
frontier = m7.compute_efficient_frontier(mu, sigma)
plotter.plot_efficient_frontier(frontier, opt_result)
plt.show()

In [ ]:
# Monte Carlo simulation
mc_result = m7.monte_carlo_simulation(
    weights=pd.Series(opt_result['weights']),
    mu=mu,
    sigma=sigma,
    n_sims=5000,
    horizon_days=252
)
print('Monte Carlo Simulation (252-day horizon, 5000 paths):')
print(f'  Median outcome: {mc_result["percentiles"][50]:.1%}')
print(f'  5th percentile: {mc_result["percentiles"][5]:.1%}')
print(f'  95th percentile: {mc_result["percentiles"][95]:.1%}')
print(f'  P(loss): {mc_result["probability_of_loss"]:.1%}')
print('\n⚠ IMPORTANT: This is a SIMULATION, NOT a PREDICTION.')
print('  The real future may be completely outside this simulated range.')

## Step 8: Full Integration — Composite Score

**Mathematical framework:**
```
Score_composite = Σ wᵢ × Mᵢ  (equal weights: wᵢ = 1/6)
Score_adjusted  = Score_composite × (1 - λ × RiskPenalty)  (λ = 0.3)
```


In [ ]:
from src.integration.aggregator import ScoreAggregator

agg = ScoreAggregator(config)

print(f'\n{"COMPOSITE SCORES":=^60}')
all_scores = {}

for ticker in TICKERS:
    module_scores = {
        'M1_Fundamentals': m1_results.get(ticker, type('R', (), {'normalised_score': None})()).normalised_score,
        'M3_TimeSeries': m3_results.get(ticker, type('R', (), {'normalised_score': None})()).normalised_score,
        'M4_Factors': m4_results.get(ticker, type('R', (), {'normalised_score': None})()).normalised_score,
    }
    risk_score = m5_results.get(ticker, type('R', (), {'normalised_score': 50})()).normalised_score
    
    result = agg.run(module_scores, risk_score=risk_score or 50)
    all_scores[ticker] = result
    
    print(f'\n{ticker}:')
    print(f'  Composite Score: {result.adjusted_score:.1f} / 100')
    print(f'  90% CI: [{result.score_ci_lower:.1f}, {result.score_ci_upper:.1f}]')

print('\n' + '='*60)

## Step 9: Ablation Study

**What happens when each module is removed?**
This tests whether every module contributes independent information.


In [ ]:
from src.comparison.ablation import AblationStudy

ablation = AblationStudy(config)

# Use first ticker for demonstration
ticker = TICKERS[0]
module_scores = {
    'M1_Fundamentals': m1_results.get(ticker, type('R', (), {'normalised_score': 50})()).normalised_score or 50,
    'M3_TimeSeries': m3_results.get(ticker, type('R', (), {'normalised_score': 50})()).normalised_score or 50,
    'M4_Factors': m4_results.get(ticker, type('R', (), {'normalised_score': 50})()).normalised_score or 50,
}
risk_score = m5_results.get(ticker, type('R', (), {'normalised_score': 50})()).normalised_score or 50

ablation_results = ablation.run_module_ablation(
    module_scores=module_scores,
    risk_score=risk_score,
    aggregator=agg
)

print(f'Ablation Study for {ticker}:')
print(ablation.explain_ablation(ablation_results))

## Step 10: Correlation Matrix

**Question**: How correlated are these assets?
Low correlation = better diversification in the portfolio.


In [ ]:
returns_df = pd.DataFrame({
    ticker: log_returns(price_data[ticker]['Adj Close'])
    for ticker in TICKERS if ticker in price_data
}).dropna()

plotter.plot_correlation_matrix(returns_df, 'Asset Return Correlation Matrix')
plt.show()

print('\nCorrelation Matrix:')
print(returns_df.corr().round(3))

## Educational Summary

### What you have just done:

1. **Loaded and validated** financial data for multiple companies
2. **Measured business quality** (Piotroski F-Score, Altman Z-Score, DuPont decomposition)
3. **Analysed time-series properties** (momentum, Hurst exponent, autocorrelation)
4. **Decomposed returns** into systematic factors (market, size, value, momentum)
5. **Quantified risk** across multiple dimensions (volatility, drawdown, VaR, CVaR, beta)
6. **Optimised a portfolio** using the Markowitz framework (mean-variance optimisation)
7. **Quantified uncertainty** using Monte Carlo simulation and bootstrap confidence intervals
8. **Tested robustness** using ablation analysis

### Critical Limitations:

- **Past ≠ Future**: All analysis is based on historical data. Markets can change.
- **Model risk**: Every model makes assumptions that may be wrong.
- **Survivorship bias**: This analysis only covers companies that still exist.
- **Transaction costs**: Real investing involves costs not fully captured here.
- **This is not financial advice**: Always consult a qualified professional.
